# 🔥 Notebook 2: Cascading Failure — Breaker vs. No Breaker

We'll show what happens when service **A** depends on service **B**, B is slow, and many clients pile up.
Without a breaker, A's threads are all blocked waiting for B → A itself goes down.
With a breaker, A fails fast and stays healthy.

## 🛠️ Setup

```bash
cd 05-microservices/circuit-breaker
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
import time, threading
from concurrent.futures import ThreadPoolExecutor

def slow_b():
    time.sleep(1.0)  # B is sick and slow
    raise RuntimeError('B timeout')

# --- A without breaker: every call to A blocks for 1s then fails ---
def a_no_breaker():
    try: slow_b()
    except Exception as e: return f'A failed: {e}'

# --- A with breaker ---
class CB:
    def __init__(self, thr=5, reset=2):
        self.state='CLOSED'; self.f=0; self.opened=0; self.thr=thr; self.reset=reset
        self.lock = threading.Lock()
    def call(self, fn):
        with self.lock:
            if self.state=='OPEN' and time.time()-self.opened<self.reset:
                raise RuntimeError('fast-fail: circuit OPEN')
            if self.state=='OPEN': self.state='HALF_OPEN'
        try:
            r = fn()
            with self.lock: self.state='CLOSED'; self.f=0
            return r
        except Exception:
            with self.lock:
                self.f+=1
                if self.f>=self.thr:
                    self.state='OPEN'; self.opened=time.time()
            raise

cb = CB(thr=5, reset=10)
def a_with_breaker():
    try: return cb.call(slow_b)
    except Exception as e: return f'A: {e}'


In [ ]:
def run_load(handler, n=10, workers=2):
    """Small worker pool = requests queue, just like real servers under pressure."""
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(lambda _: handler(), range(n)))
    return time.time()-t0

print(f'no breaker: {run_load(a_no_breaker):.2f}s total for 10 reqs (each blocks 1s on slow B)')
print(f'breaker:    {run_load(a_with_breaker):.2f}s total for 10 reqs (opens after 5, then fast-fails)')


### Why it matters
- Without breaker: A's worker threads pile up → A becomes unresponsive → *upstream* callers suffer → failure cascades across the whole system.
- With breaker: A replies quickly (with an error) and frees threads to handle other, unrelated work.

### Tips
- Combine with **fallbacks** (cached data, default values) for better UX.
- Tune threshold + reset based on normal failure rate; too sensitive = flapping.